In [125]:
import polars as pl

df = pl.scan_parquet("NF-CSE-CIC-IDS2018-V2.parquet")

In [126]:
#форму датасета (количество строк и колонок);
n_rows = df.select(pl.len()).collect().item()
n_cols = len(df.collect_schema())

print(f"Cтроки: {n_rows}, Колонки: {n_cols}")

Cтроки: 17129715, Колонки: 43


In [127]:
#типы колонок (последние пять)
schema = df.collect_schema()
last_5 = list(schema.items())[-5:]
for col, dtype in last_5:
    print(f"{col}: {dtype}")

DNS_QUERY_TYPE: Int16
DNS_TTL_ANSWER: Int32
FTP_COMMAND_RET_CODE: Int8
Label: Int8
Attack: String


In [128]:
#уникальные значения в колонке Label
df.select("Label").unique().collect()

Label
i8
0
1


In [129]:
#уникальные значения в колонке Attack
df.select("Attack").unique().collect()

Attack
str
"""DoS attacks-SlowHTTPTest"""
"""Benign"""
"""SQL Injection"""
"""Infilteration"""
"""DoS attacks-GoldenEye"""
…
"""Brute Force -Web"""
"""DoS attacks-Slowloris"""
"""FTP-BruteForce"""


In [130]:
#Распределение по меткам
result = (
    df.group_by("Label")
      .agg(pl.len().alias("count")) #использовается len() с алиасом
      .with_columns([
          (pl.col("count") / pl.col("count").sum() * 100)
          .alias("percentage")
      ])
      .collect()
)

benign = result.filter(pl.col("Label") == 0)
attack = result.filter(pl.col("Label") == 1)

print("Кол-во записей(count) и процент(percentage) 'Benign (0)':")
print(benign)

print("\nКол-во записей(count) и процент(percentage) 'Attack (1)':")
print(attack)

Кол-во записей(count) и процент(percentage) 'Benign (0)':
shape: (1, 3)
┌───────┬──────────┬────────────┐
│ Label ┆ count    ┆ percentage │
│ ---   ┆ ---      ┆ ---        │
│ i8    ┆ u32      ┆ f64        │
╞═══════╪══════════╪════════════╡
│ 0     ┆ 15101685 ┆ 88.160749  │
└───────┴──────────┴────────────┘

Кол-во записей(count) и процент(percentage) 'Attack (1)':
shape: (1, 3)
┌───────┬─────────┬────────────┐
│ Label ┆ count   ┆ percentage │
│ ---   ┆ ---     ┆ ---        │
│ i8    ┆ u32     ┆ f64        │
╞═══════╪═════════╪════════════╡
│ 1     ┆ 2028030 ┆ 11.839251  │
└───────┴─────────┴────────────┘


In [131]:
#Создание бинарного признака
df = df.with_columns(
    pl.when(pl.col("Label") == 0)
      .then(0)
      .otherwise(1)
      .cast(pl.Int8) #присваиваем тип
      .alias("is_attack")
)
#Проверяем
df.select(["Label", "is_attack"]).limit(10).collect()

Label,is_attack
i8,i8
1,1
0,0
0,0
0,0
1,1
0,0
0,0
0,0
1,1


In [132]:
#распределение по PROTOCOL только для Benign
df.filter(pl.col("Label") == 0) \
  .group_by("PROTOCOL") \
  .agg(pl.len().alias("count")) \
  .sort("count", descending=True) \
  .collect()


PROTOCOL,count
i8,u32
17,7688529
6,7406897
1,4537
2,883
58,836
47,3


In [133]:
#Агрегация по типам атак
attack_stats = (
    df.filter(pl.col("Label") != 0)  # только атаки
      .group_by("Attack") #Сгруппируйте данные по колонке Attack
      .agg([
          pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_duration"),
          pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
          pl.len().alias("total_records")
      ])
      .sort("avg_in_bytes", descending=True)
      .collect()
)

print(attack_stats)

shape: (14, 4)
┌──────────────────────────┬───────────────┬──────────────┬───────────────┐
│ Attack                   ┆ avg_duration  ┆ avg_in_bytes ┆ total_records │
│ ---                      ┆ ---           ┆ ---          ┆ ---           │
│ str                      ┆ f64           ┆ f64          ┆ u32           │
╞══════════════════════════╪═══════════════╪══════════════╪═══════════════╡
│ DDOS attack-LOIC-UDP     ┆ 4.1959e6      ┆ 5.8540e6     ┆ 2112          │
│ DDoS attacks-LOIC-HTTP   ┆ 3.7241e6      ┆ 25991.904925 ┆ 207078        │
│ Brute Force -XSS         ┆ 3.7568e6      ┆ 16871.281553 ┆ 927           │
│ Brute Force -Web         ┆ 3.7553e6      ┆ 9797.653756  ┆ 2143          │
│ SSH-Bruteforce           ┆ 733291.768981 ┆ 5828.140747  ┆ 94979         │
│ …                        ┆ …             ┆ …            ┆ …             │
│ Infilteration            ┆ 234910.506514 ┆ 791.858613   ┆ 115513        │
│ Bot                      ┆ 1.2340e6      ┆ 677.354225   ┆ 28033        

In [134]:
#Топ-3 атак по трафику
top3_attacks = attack_stats.head(3)
print(top3_attacks)

shape: (3, 4)
┌────────────────────────┬──────────────┬──────────────┬───────────────┐
│ Attack                 ┆ avg_duration ┆ avg_in_bytes ┆ total_records │
│ ---                    ┆ ---          ┆ ---          ┆ ---           │
│ str                    ┆ f64          ┆ f64          ┆ u32           │
╞════════════════════════╪══════════════╪══════════════╪═══════════════╡
│ DDOS attack-LOIC-UDP   ┆ 4.1959e6     ┆ 5.8540e6     ┆ 2112          │
│ DDoS attacks-LOIC-HTTP ┆ 3.7241e6     ┆ 25991.904925 ┆ 207078        │
│ Brute Force -XSS       ┆ 3.7568e6     ┆ 16871.281553 ┆ 927           │
└────────────────────────┴──────────────┴──────────────┴───────────────┘


In [135]:
#общее распределение по PROTOCOL;
protocol_dist = (
    df.group_by("PROTOCOL")
      .agg(pl.len().alias("count")) #использовается len() с алиасом
      .sort("count", descending=True)
      .collect()
)

print(protocol_dist)

shape: (6, 2)
┌──────────┬─────────┐
│ PROTOCOL ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 6        ┆ 9346287 │
│ 17       ┆ 7776756 │
│ 1        ┆ 4857    │
│ 2        ┆ 976     │
│ 58       ┆ 836     │
│ 47       ┆ 3       │
└──────────┴─────────┘


In [136]:
#распределение по PROTOCOL только для Attack (с разбивкой по Attack);
df.filter(pl.col("Label") != 0) \
  .group_by(["Attack", "PROTOCOL"]) \
  .agg(pl.len().alias("count")) \
  .sort("count", descending=True) \
  .collect()

Attack,PROTOCOL,count
str,i8,u32
"""DDOS attack-HOIC""",6,1066881
"""DoS attacks-Hulk""",6,432648
"""DDoS attacks-LOIC-HTTP""",6,189750
"""SSH-Bruteforce""",6,94979
"""Infilteration""",17,68676
…,…,…
"""SQL Injection""",6,432
"""Infilteration""",1,320
"""Infilteration""",2,93


In [137]:
#сравнение PROTOCOL между Benign и Attack.
comparison = (
    df.with_columns(
        pl.when(pl.col("Label") == 0)
          .then(pl.lit("Benign")) # используем pl.lit()
          .otherwise(pl.lit("Attack")) # используем pl.lit()
          .alias("TrafficType")
    )
    .group_by(["PROTOCOL", "TrafficType"])
    .agg(pl.len().alias("count"))
    .collect()
)

comparison_pivot = (
    comparison.pivot(
        values="count",
        index="PROTOCOL",
        on="TrafficType"
    )
    .fill_null(0)
)

print(comparison_pivot)

shape: (6, 3)
┌──────────┬─────────┬─────────┐
│ PROTOCOL ┆ Attack  ┆ Benign  │
│ ---      ┆ ---     ┆ ---     │
│ i8       ┆ u32     ┆ u32     │
╞══════════╪═════════╪═════════╡
│ 1        ┆ 320     ┆ 4537    │
│ 47       ┆ 0       ┆ 3       │
│ 17       ┆ 88227   ┆ 7688529 │
│ 6        ┆ 1939390 ┆ 7406897 │
│ 2        ┆ 93      ┆ 883     │
│ 58       ┆ 0       ┆ 836     │
└──────────┴─────────┴─────────┘


In [138]:
#Сравнение метрик: Benign vs Attack
result = (
    df.with_columns(
        (pl.col("Label") != 0)
        .cast(pl.Int8)
        .alias("is_attack")
    )
    .group_by("is_attack")
    .agg([
        pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
        pl.col("OUT_BYTES").mean().alias("avg_out_bytes"),
        pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_duration"),
        pl.len().alias("total_records")
    ])
    .collect()
)

#округлим для читаемости
result = result.with_columns([
    pl.col("avg_in_bytes").round(2),
    pl.col("avg_out_bytes").round(2),
    pl.col("avg_duration").round(2)
])

print(result)

shape: (2, 5)
┌───────────┬──────────────┬───────────────┬──────────────┬───────────────┐
│ is_attack ┆ avg_in_bytes ┆ avg_out_bytes ┆ avg_duration ┆ total_records │
│ ---       ┆ ---          ┆ ---           ┆ ---          ┆ ---           │
│ i8        ┆ f64          ┆ f64           ┆ f64          ┆ u32           │
╞═══════════╪══════════════╪═══════════════╪══════════════╪═══════════════╡
│ 0         ┆ 867.64       ┆ 8253.54       ┆ 185715.81    ┆ 15101685      │
│ 1         ┆ 10013.79     ┆ 2301.41       ┆ 3.7472e6     ┆ 2028030       │
└───────────┴──────────────┴───────────────┴──────────────┴───────────────┘


In [139]:
df_features = (
    df.with_columns([
        #производные признаки
        (pl.col("IN_BYTES") / (pl.col("OUT_BYTES") + 1)).alias("bytes_ratio"),
        (pl.col("IN_BYTES") + pl.col("OUT_BYTES")).alias("total_bytes"),
        (
            (pl.col("IN_BYTES") + pl.col("OUT_BYTES")) /
            (pl.col("IN_PKTS") + pl.col("OUT_PKTS") + 1)
        ).alias("packet_size_avg")
    ])
    .with_columns([
        #эвристика
        (
            (pl.col("bytes_ratio") > 10) &
            (pl.col("FLOW_DURATION_MILLISECONDS") < 500) &
            (pl.col("IN_PKTS") > 10)
        )
        .cast(pl.Int8)
        .alias("is_suspicious")
    ])
)

result = (
    df_features
    .select([
        # всего подозрительных
        pl.col("is_suspicious").sum().alias("total_flagged"),

        # из них реальные атаки
        (
            (pl.col("is_suspicious") == 1) &
            (pl.col("Label") != 0)
        ).sum().alias("true_attacks")
    ])
    .with_columns([
        # точность
        (pl.col("true_attacks") / pl.col("total_flagged"))
        .alias("precision")
    ])
    .collect()
)

print(result)

shape: (1, 3)
┌───────────────┬──────────────┬───────────┐
│ total_flagged ┆ true_attacks ┆ precision │
│ ---           ┆ ---          ┆ ---       │
│ i64           ┆ u32          ┆ f64       │
╞═══════════════╪══════════════╪═══════════╡
│ 4645          ┆ 3707         ┆ 0.798062  │
└───────────────┴──────────────┴───────────┘


In [ ]:
#Сохраняем итоговую агрегацию (по типам атак) в файл
attack_stats.write_parquet("attack_summary_by_type.parquet")